# Notebook 06 — Clasificación Supervisada con Random Forest
**Teledetección — Maestría en Ingeniería | Universidad del Magdalena**  
**Sesión 3 | Viernes 24 Jul 2026**

### ¿Qué aprenderemos?

1. Diferencia entre clasificación **supervisada** y **no supervisada**
2. Definir **Regiones de Interés (ROIs)** como muestras de entrenamiento
3. Entrenar un clasificador **Random Forest** con GEE
4. Producir un **mapa de uso del suelo** del Norte del Magdalena
5. Validar con **matriz de confusión** y calcular el **índice Kappa**

**Contexto:** Vamos a mapear 5 clases en la zona bananera y cacaotera: agua, suelo desnudo, pastizal, cultivos (banano/cacao) y bosque.

---

## Parte 0 — Instalación

In [ ]:
!pip install geemap --quiet
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import seaborn as sns

ee.Authenticate()
ee.Initialize(project='tu-proyecto-gee')

## Parte 1 — Conceptos clave (repaso rápido)

### Supervisada vs No Supervisada

| | No supervisada (S2) | Supervisada (hoy) |
|---|---|---|
| Muestras | El algoritmo agrupa píxeles similares | El analista dibuja ROIs por clase |
| Clases | Automáticas (el número N lo defines tú) | Definidas por el analista |
| Algoritmo | K-means, ISODATA | Random Forest, SVM, MaxLikelihood |
| Validación | Visual + interpretación | Matriz de confusión + Kappa |

### Random Forest para imágenes satelitales

Un RF entrena cientos de **árboles de decisión** sobre muestras aleatorias de los datos de entrenamiento (bandas = variables predictoras). Cada árbol "vota" por una clase; gana la mayoría.

Para imagen satelital:
- **Variables de entrada (features):** valores de reflectancia en cada banda + índices espectrales
- **Variable de salida (clase):** etiqueta de cobertura dibujada en los ROIs
- **El modelo aprende:** qué combinación de bandas caracteriza cada tipo de cobertura

## Parte 2 — Preparar imagen Sentinel-2 con múltiples bandas

In [ ]:
# Zona de estudio: Norte del Magdalena
zona_estudio = ee.Geometry.Rectangle([-74.40, 10.50, -73.90, 11.10])

def enmascarar_nubes(imagen):
    scl = imagen.select('SCL')
    mascara = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return imagen.updateMask(mascara)

def agregar_indices(imagen):
    ndvi = imagen.normalizedDifference(['B8',  'B4' ]).rename('NDVI')
    ndre = imagen.normalizedDifference(['B8A', 'B5' ]).rename('NDRE')
    ndmi = imagen.normalizedDifference(['B8A', 'B11']).rename('NDMI')
    ndwi = imagen.normalizedDifference(['B3',  'B8' ]).rename('NDWI')
    evi  = imagen.expression(
        '2.5*(NIR-RED)/(NIR+6*RED-7.5*BLUE+1)',
        {'NIR': imagen.select('B8').divide(10000),
         'RED': imagen.select('B4').divide(10000),
         'BLUE':imagen.select('B2').divide(10000)}
    ).rename('EVI')
    return imagen.addBands([ndvi, ndre, ndmi, ndwi, evi])

imagen_s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_estudio)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
    .map(enmascarar_nubes)
    .map(agregar_indices)
    .median()
    .clip(zona_estudio))

# Bandas que usaremos para clasificar
bandas_clasificacion = ['B2', 'B3', 'B4', 'B5', 'B8', 'B8A', 'B11', 'NDVI', 'NDRE', 'NDMI', 'NDWI', 'EVI']
print("Imagen lista. Bandas para clasificación:", bandas_clasificacion)

## Parte 3 — Definir Regiones de Interés (ROIs)

Los ROIs son polígonos que el analista dibuja sobre áreas donde la cobertura es **conocida con certeza**. Son las "muestras de verdad de campo" del algoritmo.

**Regla de oro:** dibuja ROIs en lugares que conoces o que puedes verificar en Google Earth / imagen de alta resolución. ROIs mal etiquetados degradan fuertemente el clasificador.

Las 5 clases del Norte del Magdalena:

In [ ]:
# Clases de cobertura
CLASES = {
    0: 'Agua',
    1: 'Suelo desnudo',
    2: 'Pastizal',
    3: 'Cultivos (banano/cacao)',
    4: 'Bosque',
}
COLORES = {
    0: '#2196F3',  # azul — agua
    1: '#FF9800',  # naranja — suelo
    2: '#CDDC39',  # amarillo-verde — pastizal
    3: '#4CAF50',  # verde — cultivos
    4: '#1B5E20',  # verde oscuro — bosque
}

# ROIs definidos sobre el Norte del Magdalena
# (coordenadas verificadas en Google Earth)
rois_agua = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[-74.23,10.95],[-74.18,10.95],[-74.18,10.98],[-74.23,10.98]]]), {'clase': 0, 'nombre': 'Cienaga Grande'}),
    ee.Feature(ee.Geometry.Polygon([[[-74.35,10.88],[-74.30,10.88],[-74.30,10.92],[-74.35,10.92]]]), {'clase': 0, 'nombre': 'Laguna costera'}),
])

rois_suelo = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[-74.10,10.55],[-74.05,10.55],[-74.05,10.58],[-74.10,10.58]]]), {'clase': 1, 'nombre': 'Suelo arenoso'}),
    ee.Feature(ee.Geometry.Polygon([[[-74.20,10.62],[-74.15,10.62],[-74.15,10.65],[-74.20,10.65]]]), {'clase': 1, 'nombre': 'Suelo desnudo cultivado'}),
])

rois_pastizal = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[-74.15,10.70],[-74.10,10.70],[-74.10,10.73],[-74.15,10.73]]]), {'clase': 2, 'nombre': 'Pastizal bajo'}),
    ee.Feature(ee.Geometry.Polygon([[[-74.28,10.75],[-74.22,10.75],[-74.22,10.78],[-74.28,10.78]]]), {'clase': 2, 'nombre': 'Pastizal ganadero'}),
])

rois_cultivos = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[-74.32,10.85],[-74.26,10.85],[-74.26,10.90],[-74.32,10.90]]]), {'clase': 3, 'nombre': 'Zona bananera Cienaga'}),
    ee.Feature(ee.Geometry.Polygon([[[-74.05,10.60],[-73.98,10.60],[-73.98,10.65],[-74.05,10.65]]]), {'clase': 3, 'nombre': 'Cacao SNSM piedemonte'}),
    ee.Feature(ee.Geometry.Polygon([[[-74.25,10.82],[-74.20,10.82],[-74.20,10.87],[-74.25,10.87]]]), {'clase': 3, 'nombre': 'Palma africana'}),
])

rois_bosque = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[-73.95,10.80],[-73.88,10.80],[-73.88,10.85],[-73.95,10.85]]]), {'clase': 4, 'nombre': 'Bosque SNSM'}),
    ee.Feature(ee.Geometry.Polygon([[[-74.00,10.90],[-73.93,10.90],[-73.93,10.95],[-74.00,10.95]]]), {'clase': 4, 'nombre': 'Bosque ripario Magdalena'}),
])

todos_rois = rois_agua.merge(rois_suelo).merge(rois_pastizal).merge(rois_cultivos).merge(rois_bosque)
print(f"Total ROIs definidos: {todos_rois.size().getInfo()}")
print("Distribución por clase:")
for clase_id, nombre in CLASES.items():
    n = todos_rois.filter(ee.Filter.eq('clase', clase_id)).size().getInfo()
    print(f"  Clase {clase_id} ({nombre}): {n} polígonos")

In [ ]:
# Visualizar ROIs sobre la imagen
mapa_rois = geemap.Map()
mapa_rois.centerObject(zona_estudio, zoom=10)
mapa_rois.addLayer(imagen_s2, {'bands': ['B8','B4','B3'], 'min': 0, 'max': 5000, 'gamma': 1.4}, 'Falso Color NIR')

estilos = [
    (rois_agua,     '#2196F3', 'Agua'),
    (rois_suelo,    '#FF9800', 'Suelo desnudo'),
    (rois_pastizal, '#CDDC39', 'Pastizal'),
    (rois_cultivos, '#4CAF50', 'Cultivos'),
    (rois_bosque,   '#1B5E20', 'Bosque'),
]
for rois, color, nombre in estilos:
    mapa_rois.addLayer(rois, {'color': color}, f'ROI — {nombre}')

mapa_rois

## Parte 4 — Entrenamiento en GEE

GEE extrae los valores de todas las bandas en los píxeles de los ROIs y los usa para entrenar el clasificador.

In [ ]:
# Muestrear píxeles en los ROIs
muestras = imagen_s2.select(bandas_clasificacion).sampleRegions(
    collection=todos_rois,
    properties=['clase'],
    scale=10,
    tileScale=2
)

# Dividir 70% entrenamiento / 30% validación (reproducible)
muestras_con_random = muestras.randomColumn(seed=42)
entrenamiento = muestras_con_random.filter(ee.Filter.lt('random', 0.7))
validacion    = muestras_con_random.filter(ee.Filter.gte('random', 0.7))

print(f"Total muestras: {muestras.size().getInfo()}")
print(f"Entrenamiento:  {entrenamiento.size().getInfo()} (70%)")
print(f"Validación:     {validacion.size().getInfo()} (30%)")

In [ ]:
# Entrenar Random Forest en GEE
clasificador = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42
).train(
    features=entrenamiento,
    classProperty='clase',
    inputProperties=bandas_clasificacion
)

# Aplicar clasificador a la imagen completa
mapa_clasificado = imagen_s2.select(bandas_clasificacion).classify(clasificador)
print("Clasificador entrenado con 100 árboles.")
print("Variables de entrada:", bandas_clasificacion)

In [ ]:
# Visualizar mapa clasificado
paleta_clases = ['#2196F3', '#FF9800', '#CDDC39', '#4CAF50', '#1B5E20']

mapa_resultado = geemap.Map()
mapa_resultado.centerObject(zona_estudio, zoom=10)
mapa_resultado.addLayer(imagen_s2, {'bands': ['B4','B3','B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}, 'Color Natural')
mapa_resultado.addLayer(mapa_clasificado, {'min': 0, 'max': 4, 'palette': paleta_clases}, 'Clasificación RF')

# Leyenda
mapa_resultado.add_legend(
    title='Cobertura del suelo',
    labels=list(CLASES.values()),
    colors=paleta_clases
)
mapa_resultado

## Parte 5 — Validación: Matriz de Confusión y Kappa

La **matriz de confusión** compara la clase predicha por el modelo con la clase real (verificada en campo o en imágenes de alta resolución).

- **Precisión global (Overall Accuracy):** porcentaje total de píxeles correctamente clasificados
- **Índice Kappa (κ):** mide el acuerdo más allá del azar. κ > 0.8 = excelente, κ 0.6–0.8 = bueno, κ < 0.4 = pobre

In [ ]:
# Validar en GEE con las muestras de validación
validacion_clasificada = validacion.classify(clasificador)

matriz_confusion = validacion_clasificada.errorMatrix('clase', 'classification')
precision_global = matriz_confusion.accuracy().getInfo()
kappa            = matriz_confusion.kappa().getInfo()

print(f"=== Métricas de validación ===")
print(f"Precisión global (OA):  {precision_global*100:.1f}%")
print(f"Índice Kappa (κ):       {kappa:.4f}")
print()

kappa_interp = 'Excelente' if kappa > 0.8 else 'Bueno' if kappa > 0.6 else 'Moderado' if kappa > 0.4 else 'Pobre'
print(f"Interpretación Kappa:   {kappa_interp}")
print()
print("Escala de Kappa (Landis & Koch 1977):")
print("  κ > 0.80 → Casi perfecto")
print("  κ 0.60–0.80 → Substancial")
print("  κ 0.40–0.60 → Moderado")
print("  κ 0.20–0.40 → Regular")
print("  κ < 0.20 → Leve")

In [ ]:
# Visualizar la matriz de confusión
matriz_array = np.array(matriz_confusion.getInfo())

nombres_clases = list(CLASES.values())

# Normalizar por fila (recall por clase)
totales_fila = matriz_array.sum(axis=1, keepdims=True)
totales_fila[totales_fila == 0] = 1
matriz_norm = matriz_array / totales_fila

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz absoluta
sns.heatmap(matriz_array, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases, yticklabels=nombres_clases,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Matriz de confusión (conteos)', fontsize=11)
axes[0].set_xlabel('Clase predicha')
axes[0].set_ylabel('Clase real')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

# Matriz normalizada (recall)
sns.heatmap(matriz_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=nombres_clases, yticklabels=nombres_clases,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Matriz normalizada (recall por clase)', fontsize=11)
axes[1].set_xlabel('Clase predicha')
axes[1].set_ylabel('Clase real')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

fig.suptitle(f'Validación clasificación RF — OA: {precision_global*100:.1f}%  Kappa: {kappa:.3f}', fontsize=12)
plt.tight_layout()
plt.show()

## Parte 6 — Importancia de variables

¿Qué bandas aportaron más al clasificador? Random Forest puede responder esto.

In [ ]:
# Importancia de variables en GEE RF
importancias_dict = clasificador.explain().getInfo().get('importance', {})

# Ordenar de mayor a menor
bandas_ord = sorted(importancias_dict.keys(), key=lambda b: importancias_dict[b], reverse=True)
valores_ord = [importancias_dict[b] for b in bandas_ord]

colores_imp = ['#e74c3c' if b in ['NDVI','NDRE','NDMI','NDWI','EVI'] else '#3498db' for b in bandas_ord]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(bandas_ord[::-1], valores_ord[::-1], color=colores_imp[::-1], alpha=0.8)
ax.set_xlabel('Importancia relativa')
ax.set_title('Importancia de variables — Random Forest\n(rojo = índices calculados, azul = bandas espectrales originales)')
ax.grid(axis='x', alpha=0.4)

for bar, val in zip(bars, valores_ord[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print("\nObservaciones típicas:")
print("- NDRE y NDMI suelen superar a las bandas crudas: comprimen información relevante")
print("- B11 (SWIR) aporta mucho para separar suelo de vegetación")
print("- B2 (azul) es útil para detectar agua")

## Parte 7 — Validación cruzada con scikit-learn (opcional avanzado)

Si descargamos los datos de los ROIs, podemos usar scikit-learn para un análisis de validación más completo.

In [ ]:
# Descargar muestras de GEE para análisis local
muestras_locales = muestras.getInfo()
features = muestras_locales.get('features', [])

X = np.array([[f['properties'][b] for b in bandas_clasificacion] for f in features if None not in [f['properties'].get(b) for b in bandas_clasificacion]])
y = np.array([f['properties']['clase'] for f in features if None not in [f['properties'].get(b) for b in bandas_clasificacion]])

print(f"Muestras descargadas: {len(X)} píxeles, {len(bandas_clasificacion)} variables")
print("Distribución de clases:")
for clase_id, nombre in CLASES.items():
    n = np.sum(y == clase_id)
    print(f"  {nombre}: {n} píxeles ({100*n/len(y):.1f}%)")

In [ ]:
# Entrenar RF local con scikit-learn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

rf_local = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_local.fit(X_train, y_train)
y_pred = rf_local.predict(X_test)

print("=== Reporte de clasificación ===")
nombres_target = list(CLASES.values())
print(classification_report(y_test, y_pred, target_names=nombres_target))

kappa_sklearn = cohen_kappa_score(y_test, y_pred)
print(f"Índice Kappa (scikit-learn): {kappa_sklearn:.4f}")

In [ ]:
# Matriz de confusión con scikit-learn
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
            xticklabels=nombres_target, yticklabels=nombres_target,
            linewidths=0.5, ax=ax)
ax.set_title(f'Matriz de confusión — Random Forest\nKappa = {kappa_sklearn:.3f} | OA = {rf_local.score(X_test, y_test)*100:.1f}%')
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Parte 8 — Exportar mapa clasificado

In [ ]:
tarea_clasificacion = ee.batch.Export.image.toDrive(
    image=mapa_clasificado.toByte(),
    description='clasificacion_rf_norte_magdalena_2024',
    folder='Teledeteccion_S3',
    fileNamePrefix='clasificacion_rf_norte_magdalena_2024',
    region=zona_estudio,
    scale=10,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)
tarea_clasificacion.start()
print("Exportación iniciada → Google Drive / Teledeteccion_S3")
print("El GeoTIFF tendrá valores 0–4 (una banda), abre en QGIS y asigna colores manualmente.")
print()
print("Leyenda para QGIS:")
for clase_id, nombre in CLASES.items():
    print(f"  Valor {clase_id} → {nombre} ({COLORES[clase_id]})")

---
## Resumen y reflexión

### Lo que hicimos hoy
1. Preparamos imagen Sentinel-2 con 7 bandas + 5 índices espectrales
2. Definimos ROIs en 5 clases de cobertura del Norte del Magdalena
3. Entrenamos un Random Forest con 100 árboles
4. Validamos con matriz de confusión y calculamos Kappa
5. Analizamos qué variables fueron más importantes

### Para mejorar la clasificación en tu proyecto
- **Más ROIs:** 10+ polígonos por clase, bien distribuidos en el área
- **ROIs balanceados:** similar número de píxeles por clase
- **Agregar DEM:** la altitud discrimina muy bien bosque de cultivo en la SNSM
- **Agregar SAR (Sentinel-1):** la textura del radar ayuda a separar agua de suelo húmedo
- **Validar en campo:** si tienes GPS, toma puntos de verificación independientes

### Pregunta para el proyecto final
> ¿Qué 5 clases necesitas mapear en tu área de estudio? ¿Cuántos polígonos de ROI vas a dibujar?

**Prepara esta respuesta para la presentación de 2 minutos al final de la sesión.**